In [110]:
import pandas as pd
df = pd.read_csv("../archive/Combined_News_DJIA.csv")
print(df.shape)
df.head()

(1989, 27)


,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Georgia Invades South Ossetia - if Russia ge...,b'Al-Qaeda Faces Islamist Backlash',"b'Condoleezza Rice: ""The US would not act to p...",b'This is a busy day: The European Union has ...,"b""Georgia will withdraw 1,000 soldiers from Ir...",b'Why the Pentagon Thinks Attacking Iran is a ...,b'Caucasus in crisis: Georgia invades South Os...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge"""
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b'Israel and the US behind the Georgian aggres...,"b'""Do not believe TV, neither Russian nor Geor...",b'Riots are still going on in Montreal (Canada...,b'China to overtake US as largest manufacturer',b'War in South Ossetia [PICS]',b'Israeli Physicians Group Condemns State Tort...,b' Russia has just beaten the United States ov...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo..."
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'U.S. troops still in Georgia (did you know t...,b'Why Russias response to Georgia was right',"b'Gorbachev accuses U.S. of making a ""serious ...","b'Russia, Georgia, and NATO: Cold War Two'",b'Remember that adorable 62-year-old who led y...,b'War in Georgia: The Israeli connection',b'All signs point to the US encouraging Georgi...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man..."
3,2008-08-13,0,b' U.S. refuses Israel weapons to attack Iran:...,"b""When the president ordered to attack Tskhinv...",b' Israel clears troops who killed Reuters cam...,b'Britain\'s policy of being tough on drugs is...,b'Body of 14 year old found in trunk; Latest (...,b'China has moved 10 *million* quake survivors...,"b""Bush announces Operation Get All Up In Russi...",b'Russian forces sink Georgian ships ',...,b'Elephants extinct by 2020?',b'US humanitarian missions soon in Georgia - i...,"b""Georgia's DDOS came from US sources""","b'Russian convoy heads into Georgia, violating...",b'Israeli defence minister: US against strike ...,b'Gorbachev: We Had No Choice',b'Witness: Russian forces head towards Tbilisi...,b' Quarter of Russians blame U.S. for conflict...,b'Georgian president says US military will ta...,b'2006: Nobel laureate Aleksander Solzhenitsyn...
4,2008-08-14,1,b'All the experts admit that we should legalis...,b'War in South Osetia - 89 pictures made by a ...,b'Swedish wrestler Ara Abrahamian throws away ...,b'Russia exaggerated the death toll in South O...,b'Missile That Killed 9 Inside Pakistan May Ha...,"b""Rushdie Condemns Random House's Refusal to P...",b'Poland and US agree 

# Preprocessing steps (in order of execution)

1. POS tagging
2. Named entity recognition (NER)
3. Sentence segmentation
4. Word tokenization
5. Stop word removal (with exception og negation terms) and punctuation removal (with exception of financial symbols)
6. Lemmatization

In [ ]:
import string
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tree import Tree


for resource in (
    "punkt_tab",
    "wordnet",
    "stopwords",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker_tab",
    "words",
):
    nltk.download(resource, quiet=True)

lemmatizer = WordNetLemmatizer()
mystopwords = set(stopwords.words("english")) - {"no", "nor", "not", "n't"}

pos_map = {
    "J": "a",  # adjective
    "V": "v",  # verb
    "N": "n",  # noun
    "R": "r",  # adverb
}

financial_symbols = {"$", "£", "€", "¥", "%"}

def pre_process_sentence(sentence):
    # POS tagging on the original words
    tagged = nltk.pos_tag(word_tokenize(sentence))

    # NER before lowercasing or removing words
    entity_tree = nltk.ne_chunk(tagged, binary=False)
    entities = [
        {
            "text": " ".join(word for word, _ in chunk.leaves()),
            "label": chunk.label(),
        }
        for chunk in entity_tree
        if isinstance(chunk, Tree)
    ]

    # Remove punctuation/stopwords and lemmatize using POS
    tokens = []
    for word, tag in tagged:
        word = word.lower()

        if word in mystopwords:
            continue

        if not word.strip(string.punctuation) and word not in financial_symbols:
            continue

        wordnet_pos = pos_map.get(tag[0])
        if wordnet_pos is not None:
            word = lemmatizer.lemmatize(word, pos=wordnet_pos)

        tokens.append(word)

    return {"tokens": tokens, "entities": entities}


def pre_process_text(text):
    if not isinstance(text, str):
        return []

    if text.startswith(("b'", 'b"')) and text.endswith(text[1]):
        text = text[2:-1]

    return [
        pre_process_sentence(sentence)
        for sentence in sent_tokenize(text)
    ]

In [114]:
sent = "b\"Georgia 'downs two Russian warplanes' as countries move to brink of war\""
print(pre_process_text(sent))

[{'tokens': ['georgia', 'down', 'two', 'russian', 'warplane', 'country', 'move', 'brink', 'war'], 'entities': [{'text': 'Georgia', 'label': 'GPE'}, {'text': 'Russian', 'label': 'GPE'}]}]
